In [5]:
path_1 = r"C:\Users\vaibh\OneDrive\Desktop\Workstation\Superjoin\data\starter-datasets\delhivery\01-delhivery-prospectus-2022-excerpt.pdf"
path_2 = r"C:\Users\vaibh\OneDrive\Desktop\Workstation\Superjoin\data\starter-datasets\delhivery\02-delhivery-annual-report-fy24-excerpt.pdf"
path_3 = r"C:\Users\vaibh\OneDrive\Desktop\Workstation\Superjoin\data\starter-datasets\delhivery\03-delhivery-q4-fy24-earnings-presentation.pdf"
path_4 = r"C:\Users\vaibh\OneDrive\Desktop\Workstation\Superjoin\data\starter-datasets\india-macroeconomy\01-india-economic-survey-2024-25-excerpt.pdf"
path_5 = r"C:\Users\vaibh\OneDrive\Desktop\Workstation\Superjoin\data\starter-datasets\india-macroeconomy\02-rbi-annual-report-2024-25-excerpt.pdf"
path_6 = r"C:\Users\vaibh\OneDrive\Desktop\Workstation\Superjoin\data\starter-datasets\india-macroeconomy\03-imf-india-2025-article-iv-excerpt.pdf"

In [3]:
!pip install pymupdf

   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
    --------------------------------------- 0.3/18.7 MB ? eta -:--:--
    --------------------------------------- 0.3/18.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/18.7 MB 1.2 MB/s eta 0:00:16
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- ------------------------------------- 1.0/18.7 MB 1.2 MB/s eta 0:00:15
   -- -----------------------------------

In [6]:
# parser 

import fitz  # PyMuPDF
import json
import os
from pathlib import Path

def parse_pdf_to_chunks(pdf_path: str, chunk_size: int = 1000) -> list[dict]:
    """
    Parses a PDF file page by page using PyMuPDF and extracts text chunks 
    with their corresponding page numbers and document metadata.
    """
    doc_id = Path(pdf_path).name
    pdf_document = fitz.open(pdf_path)
    chunks = []

    for page_num in range(len(pdf_document)):
        page = pdf_document[page_num]
        text = page.get_text("text")
        
        # Basic cleanup of extra whitespaces
        cleaned_text = " ".join(text.split())
        
        if not cleaned_text:
            continue

        # Optional: Split large pages into smaller character-length chunks if needed,
        # or keep the page text as the base unit for fact extraction.
        chunks.append({
            "document_id": doc_id,
            "page_no": page_num + 1,
            "text": cleaned_text
        })
        
    pdf_document.close()
    return chunks

def save_chunks_to_json(chunks: list[dict], output_dir: str = "data/processed"):
    """Saves parsed chunks locally for downstream LLM extraction."""
    os.makedirs(output_dir, exist_ok=True)
    if not chunks:
        return
        
    doc_id = chunks[0]["document_id"]
    output_path = os.path.join(output_dir, f"{doc_id}_chunks.json")
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)
        
    print(f"Successfully processed {len(chunks)} pages from {doc_id}. Saved to {output_path}")

# Example usage:
if __name__ == "__main__":
    # Ensure you have 'PyMuPDF' installed: pip install pymupdf
    sample_pdf = path_1
    
    if os.path.exists(sample_pdf):
        extracted_chunks = parse_pdf_to_chunks(sample_pdf)
        save_chunks_to_json(extracted_chunks)
    else:
        print(f"Place a PDF at '{sample_pdf}' to test the script.")

Successfully processed 100 pages from 01-delhivery-prospectus-2022-excerpt.pdf. Saved to data/processed\01-delhivery-prospectus-2022-excerpt.pdf_chunks.json
